# Car AI Project - Data Gathering

#### Introduction

This notebook represents the entry point of the project pipeline and focuses on the controlled ingestion of the raw automotive dataset into the analytics environment.

The objective of this stage is to load the original CSV data in a reproducible, traceable, and model-agnostic way while ensuring schema consistency and storage reliability for downstream processing.

At this stage:

- no cleaning is performed
- no transformations are applied
- no feature engineering is introduced
- no modeling assumptions are made

The dataset is validated structurally, standardized where necessary for platform compatibility, and stored in Delta Lake format to support scalable processing, reproducibility, and experiment tracking throughout the project lifecycle.

## 1. Libraries

In [0]:
import pandas as pd
from datetime import datetime


In [0]:
# Import project configuration
import sys
import os

# Add parent directory to path to import config
sys.path.append("..")
from config import *


print("PROJECT CONFIGURATION LOADED")

print(f"\nBASE_PATH: {BASE_PATH}")
print(f"\nData Paths:")
print(f"   - RAW_DATA_PATH: {RAW_DATA_PATH}")
print(f"   - FEATURES_PATH: {FEATURES_PATH}")
print(f"   - PROCESSED_DATA_PATH: {PROCESSED_DATA_PATH}")
print(f"   - TRAIN_TEST_PATH: {TRAIN_TEST_PATH}")
print(f"   - METRICS_PATH: {METRICS_PATH}")
print(f"\nModel Path:")
print(f"   - MODEL_PATH: {MODEL_PATH}")
print(f"\nUnity Catalog:")
print(f"   - SOURCE_CSV_FILE: {SOURCE_CSV_FILE}")
print(f"   - RAW_CARS_TABLE: {RAW_CARS_TABLE}")
print(f"   - CLEANED_CARS_TABLE: {CLEANED_CARS_TABLE}")


PROJECT CONFIGURATION LOADED

BASE_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject

Data Paths:
   - RAW_DATA_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/raw
   - FEATURES_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/features
   - PROCESSED_DATA_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/processed
   - TRAIN_TEST_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/train_test
   - METRICS_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/metrics

Model Path:
   - MODEL_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/models

Unity Catalog:
   - SOURCE_CSV_FILE: /Volumes/workspace/caraiproject/caraiproject/Cars_Datasets_2025.csv
   - RAW_CARS_TABLE: workspace.caraiproject.raw_cars_data_gathered
   - CLEANED_CARS_TABLE: workspace.caraiproject.cleaned_cars_data


## 2. Load Dataset

In [0]:
# Create widget for dataset filename (if not exists)
dbutils.widgets.text("dataset", "cars_dataset.csv", "Dataset Filename")

# Get widget value
dataset = dbutils.widgets.get("dataset")

In [0]:
# Load the Cars Dataset from Unity Catalog Volumes
# Use config constant for source file path
file_path = SOURCE_CSV_FILE

# Read CSV file into pandas DataFrame (with encoding for special characters)
df = pd.read_csv(file_path, encoding='latin-1')

# Display basic information
print(f"Dataset loaded successfully!")
print(f"Source: {file_path}")
print(f"Number of rows: {len(df):,}")
print(f"Number of columns: {len(df.columns)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:")
print(df.dtypes)

# Show first few rows
print(f"\nFirst 5 rows:")
display(df.head())

Dataset loaded successfully!
Source: /Volumes/workspace/caraiproject/caraiproject/Cars_Datasets_2025.csv
Number of rows: 1,218
Number of columns: 11

Columns: ['Company Names', 'Cars Names', 'Engines', 'CC/Battery Capacity', 'HorsePower', 'Total Speed', 'Performance(0 - 100 )KM/H', 'Cars Prices', 'Fuel Types', 'Seats', 'Torque']

Data types:
Company Names                object
Cars Names                   object
Engines                      object
CC/Battery Capacity          object
HorsePower                   object
Total Speed                  object
Performance(0 - 100 )KM/H    object
Cars Prices                  object
Fuel Types                   object
Seats                        object
Torque                       object
dtype: object

First 5 rows:


Company Names,Cars Names,Engines,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Fuel Types,Seats,Torque
FERRARI,SF90 STRADALE,V8,3990 cc,963 hp,340 km/h,2.5 sec,"$1,100,000",plug in hyrbrid,2,800 Nm
ROLLS ROYCE,PHANTOM,V12,6749 cc,563 hp,250 km/h,5.3 sec,"$460,000",Petrol,5,900 Nm
Ford,KA+,1.2L Petrol,"1,200 cc",70-85 hp,165 km/h,10.5 sec,"$12,000-$15,000",Petrol,5,100 - 140 Nm
MERCEDES,GT 63 S,V8,"3,982 cc",630 hp,250 km/h,3.2 sec,"$161,000",Petrol,4,900 Nm
AUDI,AUDI R8 Gt,V10,"5,204 cc",602 hp,320 km/h,3.6 sec,"$253,290",Petrol,2,560 Nm


###2.1 Validate Dataset Structure

In [0]:
# Validate Dataset Structure
print("DATASET STRUCTURE VALIDATION")
# 1. Define expected columns
expected_columns = [
    'Company Names',
    'Cars Names',
    'Engines',
    'CC/Battery Capacity',
    'HorsePower',
    'Total Speed',
    'Performance(0 - 100 )KM/H',
    'Cars Prices',
    'Fuel Types',
    'Seats',
    'Torque'
]

# 2. Check if expected columns exist
print("\n1. CHECKING EXPECTED COLUMNS...")
missing_columns = [col for col in expected_columns if col not in df.columns]
if missing_columns:
    print(f"   FAIL: Missing columns: {missing_columns}")
else:
    print(f"   PASS: All expected columns are present")

# 3. Check for unexpected extra columns
print("\n2. CHECKING FOR UNEXPECTED COLUMNS...")
extra_columns = [col for col in df.columns if col not in expected_columns]
if extra_columns:
    print(f"   WARNING: Found unexpected columns: {extra_columns}")
else:
    print(f"   PASS: No unexpected columns found")

# 4. Validate data types (pandas auto-infers types)
print("\n3. VALIDATING DATA TYPES...")
print("   Current data types:")
for col in df.columns:
    print(f"      - {col}: {df[col].dtype}")
print(f"   PASS: Data types inferred by pandas")

# 5. Basic sanity checks
print("\n4. PERFORMING BASIC SANITY CHECKS...")
validation_issues = []

# Count total missing values
total_missing = df.isnull().sum().sum()
if total_missing > 0:
    print(f"   INFO: Found {total_missing:,} total missing values across all columns")
    print(f"   Missing values by column:")
    missing_by_col = df.isnull().sum()
    for col, count in missing_by_col[missing_by_col > 0].items():
        print(f"      - {col}: {count} ({count/len(df)*100:.1f}%)")

# Count duplicates
duplicates = df.duplicated().sum()
if duplicates > 0:
    print(f"   INFO: Found {duplicates} duplicate rows (not removed at this stage)")

DATASET STRUCTURE VALIDATION

1. CHECKING EXPECTED COLUMNS...
   PASS: All expected columns are present

2. CHECKING FOR UNEXPECTED COLUMNS...
   PASS: No unexpected columns found

3. VALIDATING DATA TYPES...
   Current data types:
      - Company Names: object
      - Cars Names: object
      - Engines: object
      - CC/Battery Capacity: object
      - HorsePower: object
      - Total Speed: object
      - Performance(0 - 100 )KM/H: object
      - Cars Prices: object
      - Fuel Types: object
      - Seats: object
      - Torque: object
   PASS: Data types inferred by pandas

4. PERFORMING BASIC SANITY CHECKS...
   INFO: Found 10 total missing values across all columns
   Missing values by column:
      - CC/Battery Capacity: 3 (0.2%)
      - Performance(0 - 100 )KM/H: 6 (0.5%)
      - Torque: 1 (0.1%)
   INFO: Found 4 duplicate rows (not removed at this stage)


In [0]:
# Rename columns to snake_case (using pandas) to save the dataframe in a Delta table
print("\nRenaming columns to be Delta-compatible...")
column_mapping = {
    'Company Names': 'company_names',
    'Cars Names': 'cars_names',
    'Engines': 'engines',
    'CC/Battery Capacity': 'cc_battery_capacity',
    'HorsePower': 'horsepower',
    'Total Speed': 'total_speed',
    'Performance(0 - 100 )KM/H': 'performance_0_100_kmh',
    'Cars Prices': 'cars_prices',
    'Fuel Types': 'fuel_types',
    'Seats': 'seats',
    'Torque': 'torque'
}

# Create a copy with renamed columns
df_renamed = df.rename(columns=column_mapping)
print("Columns renamed to snake_case format")



Renaming columns to be Delta-compatible...
Columns renamed to snake_case format


## 3. Export the Dataframe into a Delta Table

In [0]:
# Export the raw dataset to Delta format in Unity Catalog
print("EXPORTING DATASET TO DELTA TABLE")

# Use config constant for target table
full_table_name = RAW_CARS_TABLE

print(f"\nTarget table: {full_table_name}")

# Convert pandas DataFrame to Spark DataFrame for Delta write
spark_df = spark.createDataFrame(df_renamed)

# Write to Delta table
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(full_table_name)

# Add table comment and properties
spark.sql(f"""
    COMMENT ON TABLE {full_table_name} IS 
    'Raw cars dataset imported from CSV using pandas. Contains unprocessed data for 1,218 vehicles including specifications, performance metrics, and pricing information. No cleaning or transformations applied. Column names converted to snake_case for Delta compatibility.'
""")

spark.sql(f"""
    ALTER TABLE {full_table_name} SET TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'true',
        'project' = 'CarAIProject',
        'pipeline_stage' = 'raw',
        'data_classification' = 'internal',
        'processing_framework' = 'pandas'
    )
""")

# Verify the table
final_row_count = spark.table(full_table_name).count()

print("\nEXPORT COMPLETED SUCCESSFULLY")
print(f"\nRaw dataset exported to: {full_table_name}")
print(f"Processing: pandas DataFrame -> Spark DataFrame -> Delta Table")
print(f"Rows exported: {final_row_count:,}")
print(f"Ready for downstream processing in cleaning/transformation pipeline")

EXPORTING DATASET TO DELTA TABLE

Target table: workspace.caraiproject.raw_cars_data_gathered

EXPORT COMPLETED SUCCESSFULLY

Raw dataset exported to: workspace.caraiproject.raw_cars_data_gathered
Processing: pandas DataFrame -> Spark DataFrame -> Delta Table
Rows exported: 1,218
Ready for downstream processing in cleaning/transformation pipeline
